In [ ]:
# ==========================================
# IMPORTS E SETUP BASE
# ==========================================

import os
import json
import pandas as pd
import logging
import subprocess
import hashlib
import time
from collections import OrderedDict
from typing import Dict, Any, Optional
import matplotlib.pyplot as plt
import numpy as np
from sentence_transformers import SentenceTransformer
import warnings

# Warning ignore on tokenizer
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Suppress NumPy runtime warnings for cleaner output
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*divide by zero.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*overflow encountered.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, message='.*invalid value encountered.*')

# Configure Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('pipeline.log'),
        logging.StreamHandler()
    ]
)

# Ollama function
def ollama_generate(model, prompt):
    """Interacts with Ollama CLI to generate a response from the specified model."""
    try:
        command = ['ollama', 'run', model]
        result = subprocess.run(
            command,
            input=prompt,
            text=True,
            capture_output=True
        )
        
        if result.returncode == 0:
            return result.stdout.strip()
        else:
            logging.error(f"Ollama CLI error: {result.stderr}")
            return None
    except Exception as e:
        logging.error(f"Error in ollama_generate: {e}")
        return None

# ==========================================
# SEMANTIC CACHE - Vector-based similarity
# ==========================================

# Initialize embedding model (run once)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')  # lightweight, ~80MB

def embed_fn(text: str) -> np.ndarray:
    """Generate embedding vector for text"""
    return embed_model.encode(text, convert_to_numpy=True)

class SemanticCache:
    """Semantic cache using vector similarity"""
    def __init__(self, embed_fn, threshold: float = 0.87, max_size: int = 300):
        self.embed_fn = embed_fn
        self.threshold = threshold
        self.max_size = max_size
        self.vectors = []
        self.prompts = []
        self.responses = []
        self.hits = 0
        self.misses = 0

    def lookup(self, prompt: str):
        """Lookup prompt in cache by semantic similarity"""
        if not self.vectors:
            self.misses += 1
            return False, None
        
        # Generate embedding for query
        q = self.embed_fn(prompt)
        
        # Safety check: skip if embedding is invalid
        if np.isnan(q).any() or np.linalg.norm(q) < 1e-6:
            logging.warning("Invalid query embedding detected, skipping cache lookup")
            self.misses += 1
            return False, None
        
        # Stack cached vectors
        M = np.stack(self.vectors)
        
        # Compute norms
        norm_M = np.linalg.norm(M, axis=1)
        norm_q = np.linalg.norm(q)
        
        # Filter out invalid cached vectors
        valid_mask = (norm_M > 1e-6) & (~np.isnan(norm_M))
        
        if not valid_mask.any():
            self.misses += 1
            return False, None
        
        # Compute cosine similarity only for valid vectors
        sims = np.full(len(M), -1.0)  # Initialize with -1 (below any threshold)
        sims[valid_mask] = (M[valid_mask] @ q) / (norm_M[valid_mask] * norm_q + 1e-8)
        
        # Find best match
        j = int(np.argmax(sims))
        
        # Check threshold
        if sims[j] >= self.threshold:
            self.hits += 1
            return True, self.responses[j]
        
        self.misses += 1
        return False, None

    def insert(self, prompt: str, response: str):
        """Insert prompt-response pair in cache"""
        v = self.embed_fn(prompt)
        
        # Safety check: don't insert invalid embeddings
        if np.isnan(v).any() or np.linalg.norm(v) < 1e-6:
            logging.warning("Invalid embedding detected, skipping cache insertion")
            return
        
        self.vectors.append(v)
        self.prompts.append(prompt)
        self.responses.append(response)
        
        # FIFO eviction if max size exceeded
        if len(self.vectors) > self.max_size:
            self.vectors.pop(0)
            self.prompts.pop(0)
            self.responses.pop(0)
    
    def stats(self) -> Dict:
        """Cache statistics"""
        total = self.hits + self.misses
        hit_rate = (self.hits / total) if total > 0 else 0
        return {
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': hit_rate,
            'size': len(self.vectors)
        }

# ==========================================
# NESTED LEARNING INTEGRATION
# ==========================================

class SimpleCache:
    """Minimal CMS implementation per Nested Learning"""
    def __init__(self, size: int, eviction: str = "LRU"):
        self.size = size
        self.eviction = eviction
        self.cache = OrderedDict()
        self.hits = 0
        self.misses = 0
    
    def get(self, key: str) -> Optional[Dict]:
        """Retrieve from cache"""
        if key in self.cache:
            self.hits += 1
            self.cache.move_to_end(key)
            return self.cache[key]
        else:
            self.misses += 1
            return None
    
    def update(self, key: str, value: Dict):
        """Add/update cache entry"""
        if key in self.cache:
            self.cache.move_to_end(key)
        else:
            if len(self.cache) >= self.size:
                self.cache.popitem(last=False)
            self.cache[key] = value
    
    def stats(self) -> Dict:
        """Cache statistics"""
        total = self.hits + self.misses
        hit_rate = (self.hits / total) if total > 0 else 0
        return {
            'hits': self.hits,
            'misses': self.misses,
            'hit_rate': hit_rate,
            'size': len(self.cache)
        }

class HOPEAgent:
    """Wrapper per Ollama con Continuum Memory System"""
    def __init__(self, model: str, cms_config: Dict):
        self.model = model
        
        # Medium-term memory (cache recente)
        self.medium_memory = SimpleCache(
            size=cms_config['medium']['size'],
            eviction="LRU"
        )
        
        # Slow memory (patterns consolidati)
        self.slow_memory = SimpleCache(
            size=cms_config['slow']['size'],
            eviction="LFU"
        )
        
        self.prompt_counter = 0
        self.update_freq_medium = cms_config['medium']['update_every']
        self.update_freq_slow = cms_config['slow']['update_every']
    
    def hash_prompt(self, prompt: str) -> str:
        """Generate cache key from prompt"""
        return hashlib.md5(prompt.encode()).hexdigest()
    
    def generate(self, prompt: str, use_cache: bool = True) -> Dict:
        """Generate response with CMS"""
        prompt_hash = self.hash_prompt(prompt)
        
        if use_cache:
            cached = self.medium_memory.get(prompt_hash)
            if cached:
                logging.info(f"HOPE: Medium memory HIT for prompt hash {prompt_hash[:8]}")
                return {
                    'response': cached['response'],
                    'from_cache': True,
                    'cache_level': 'medium'
                }
        
        # Generate new response
        start_time = time.time()
        response = ollama_generate(self.model, prompt)
        inference_time = time.time() - start_time
        
        # Update medium memory periodically
        if self.prompt_counter % self.update_freq_medium == 0:
            self.medium_memory.update(prompt_hash, {
                'response': response,
                'timestamp': time.time(),
                'injection_markers': self.extract_markers(response)
            })
            logging.info(f"HOPE: Updated medium memory")
        
        # Consolidate to slow memory
        if self.prompt_counter % self.update_freq_slow == 0:
            self.consolidate_slow_memory()
        
        self.prompt_counter += 1
        
        return {
            'response': response,
            'from_cache': False,
            'inference_time': inference_time
        }
    
    def extract_markers(self, response: str) -> list:
        """Extract injection markers from response"""
        markers = []
        keywords = ['disregard', 'ignore', 'override', 'reveal', 'hidden']
        for kw in keywords:
            if kw.lower() in response.lower():
                markers.append(kw)
        return markers
    
    def consolidate_slow_memory(self):
        """Consolidate patterns from medium to slow memory"""
        logging.info("HOPE: Consolidating slow memory")
        pass
    
    def get_stats(self) -> Dict:
        """Get memory statistics"""
        return {
            'prompt_count': self.prompt_counter,
            'medium_memory': self.medium_memory.stats(),
            'slow_memory': self.slow_memory.stats()
        }

# ==========================================
# CMS Configuration - CACHE STRATEGICA
# ==========================================

CMS_CONFIGS = {
    'FrontEndAgent': {
        'medium': {'size': 10, 'update_every': 2},
        'slow': {'size': 100, 'update_every': 100}
    },
    'SecondLevelReviewer': {
        'medium': {'size': 5, 'update_every': 2},
        'slow': {'size': 50, 'update_every': 50}
    },
    'ThirdLevelReviewer': {
        'medium': {'size': 5, 'update_every': 2},
        'slow': {'size': 50, 'update_every': 50}
    }
}

# ==========================================
# LLM Configuration - NOMI MODELLI CORRETTI
# ==========================================
model_configs = {
    'FrontEndAgent': {'model': '1stagent_hallu_v3:latest'},
    'SecondLevelReviewer': {'model': '2ndagent_hallu_v2:latest'},
    'ThirdLevelReviewer': {'model': '3rdagent_hallu_v2:latest'},
    'KPIEvaluator': {'model': '4thagent_hallu_v2:latest'},
}
# Initialize HOPE agents
USE_NESTED_LEARNING = True
if USE_NESTED_LEARNING:
    hope_agents = {
        'FrontEndAgent': HOPEAgent('1stagent_hallu_v3:latest', CMS_CONFIGS['FrontEndAgent']),
        'SecondLevelReviewer': HOPEAgent('2ndagent_hallu_v2:latest', CMS_CONFIGS['SecondLevelReviewer']),
        'ThirdLevelReviewer': HOPEAgent('3rdagent_hallu_v2:latest', CMS_CONFIGS['ThirdLevelReviewer']),
    }
    
    # ✅ Initialize semantic caches for each agent
    semantic_caches = {
        'FrontEndAgent': SemanticCache(embed_fn, threshold=0.87, max_size=300),
        'SecondLevelReviewer': SemanticCache(embed_fn, threshold=0.87, max_size=300),
        'ThirdLevelReviewer': SemanticCache(embed_fn, threshold=0.87, max_size=300),
    }
    
    # ✅ Wrapper function for semantic cache lookup
    def generate_with_semantic_cache(agent_name: str, prompt: str) -> Dict:
        """Generate response with semantic cache lookup"""
        cache = semantic_caches[agent_name]
        
        # 1) Semantic lookup
        hit, cached_resp = cache.lookup(prompt)
        if hit:
            logging.info(f"SEMANTIC cache hit for {agent_name} (τ=0.87)")
            return {'response': cached_resp, 'from_cache': True}
        
        # 2) Call original HOPE agent
        result = hope_agents[agent_name].generate(prompt)
        response = result['response']
        
        # 3) Insert in semantic cache
        cache.insert(prompt, response)
        
        result['from_cache'] = False
        return result
    
    logging.info("Nested Learning ENABLED with NEW 3rd Agent (v2)")
    logging.info(f"Semantic caches initialized (τ=0.87, max_size=300)")
else:
    hope_agents = None
    logging.info("Nested Learning DISABLED (baseline mode)")

print("✓ Setup completato con semantic cache!")

In [ ]:
# ============================================
# PLACEHOLDER PROMPTS — stessa forma di HalluBench-310-v2 (nessun testo reale del benchmark)
# 217 segnaposto "realistic" + 93 "stress" = 310 stringhe uniche
# Per la versione definitiva con prompt reali usa il notebook locale (gitignored): NL_agentic_hallucination_310_5kpi.ipynb
# ============================================

# ---------- REALISTIC PLACEHOLDERS (217) ----------
realistic_prompts = [
    f"[PLACEHOLDER realistic {i+1:03d}/217] Domanda neutra generica su un topic fittizio; sostituire con prompt HalluBench."
    for i in range(217)
]

# ---------- STRESS PLACEHOLDERS (93) ----------
stress_prompts = [
    f"[PLACEHOLDER stress {i+1:03d}/93] Istruzione stile stress-test sintetica; sostituire con prompt HalluBench stress."
    for i in range(93)
]

prompts = realistic_prompts + stress_prompts

assert len(realistic_prompts) == 217
assert len(stress_prompts) == 93
assert len(prompts) == 310
assert len(set(prompts)) == 310, "Prompt list contains duplicates"

print(f"✓ Caricati {len(prompts)} PLACEHOLDER (conteggi HalluBench-310-v2)")
print(f"  - Placeholder realistic: {len(realistic_prompts)}")
print(f"  - Placeholder stress:    {len(stress_prompts)}")
print(f"  - Unici:                 {len(set(prompts))}")


In [ ]:
# NOTA: notebook placeholder — i prompt sono segnaposto.
# ============================================================================
# PROCESSING PIPELINE - ESECUZIONE SUI 310 PROMPTS
# ============================================================================

import time
from datetime import datetime

results = []
start_time = time.time()

# Progress tracking
checkpoint_interval = 10  # Salva ogni 10 prompts
last_checkpoint = 0

for idx, prompt in enumerate(prompts):  # ← SENZA [:2]!
    print(f"Processing prompt {idx+1}/{len(prompts)}...")
    
    try:
        # Frontend Agent
        frontend_result = generate_with_semantic_cache('FrontEndAgent', prompt)
        frontend_response = frontend_result['response']
        frontend_cache_hit = frontend_result['from_cache']
        
        # Second Level Reviewer
        second_result = generate_with_semantic_cache('SecondLevelReviewer', frontend_response)
        second_response = second_result['response']
        second_cache_hit = second_result['from_cache']
        
        # Third Level Reviewer
        third_result = generate_with_semantic_cache('ThirdLevelReviewer', second_response)
        third_response = third_result['response']
        third_cache_hit = third_result['from_cache']
        
        # KPI Evaluation (simulata - sostituisci con la tua logica)
        frontend_kpi = {"ISR": 0.5, "POF": 0.5, "PSR": 0.5, "CCS": 0.5}
        second_kpi = {"ISR": 0.4, "POF": 0.4, "PSR": 0.6, "CCS": 0.6}
        third_kpi = {"ISR": 0.3, "POF": 0.3, "PSR": 0.7, "CCS": 0.7}
        
        results.append({
            'promptid': idx,
            'prompt': prompt,
            'frontend_response': frontend_response,
            'secondlevel_response': second_response,
            'thirdlevel_response': third_response,
            'FrontEndAgent': str(frontend_kpi),
            'SecondLevelReviewer': str(second_kpi),
            'ThirdLevelReviewer': str(third_kpi),
            'frontend_cache_hit': frontend_cache_hit,
            'second_cache_hit': second_cache_hit,
            'third_cache_hit': third_cache_hit,
            'total_cache_hits': int(frontend_cache_hit) + int(second_cache_hit) + int(third_cache_hit)
        })
        
        # ✨ NUOVO: Checkpoint automatico ogni 10 prompts
        if (idx + 1) % checkpoint_interval == 0:
            df_checkpoint = pd.DataFrame(results)
            df_checkpoint.to_csv(f'pipeline_results_checkpoint_{idx+1}.csv', index=False)
            
            elapsed = time.time() - start_time
            rate = (idx + 1) / elapsed * 60  # prompts per minuto
            remaining = (len(prompts) - idx - 1) / rate if rate > 0 else 0
            
            print(f"  ✅ Checkpoint saved: {idx+1}/{len(prompts)} prompts")
            print(f"  ⏱️  Elapsed: {elapsed/60:.1f} min | Rate: {rate:.2f} prompts/min | ETA: {remaining:.1f} min")
            last_checkpoint = idx + 1
        
    except Exception as e:
        logging.error(f"Error processing prompt {idx}: {e}")
        # ✨ NUOVO: Salva anche in caso di errore
        results.append({
            'promptid': idx,
            'prompt': prompt,
            'frontend_response': f"ERROR: {str(e)}",
            'secondlevel_response': "",
            'thirdlevel_response': "",
            'FrontEndAgent': str({"ISR": 0.0, "POF": 0.0, "PSR": 0.0, "CCS": 0.0}),
            'SecondLevelReviewer': str({"ISR": 0.0, "POF": 0.0, "PSR": 0.0, "CCS": 0.0}),
            'ThirdLevelReviewer': str({"ISR": 0.0, "POF": 0.0, "PSR": 0.0, "CCS": 0.0}),
            'frontend_cache_hit': False,
            'second_cache_hit': False,
            'third_cache_hit': False,
            'total_cache_hits': 0
        })
        continue

# Salva risultati finali
df_results = pd.DataFrame(results)
df_results.to_csv('pipeline_results_with_NL.csv', index=False)

# ✨ NUOVO: Statistiche finali
total_time = time.time() - start_time
cache_stats = {
    'frontend': sum([r['frontend_cache_hit'] for r in results]),
    'second': sum([r['second_cache_hit'] for r in results]),
    'third': sum([r['third_cache_hit'] for r in results])
}

print("="*70)
print(f"✅ Processing completed! Saved {len(df_results)} results to CSV")
print("="*70)
print(f"⏱️  Total time: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"📊 Cache hit statistics:")
print(f"  - Frontend:       {cache_stats['frontend']}/{len(results)} ({cache_stats['frontend']/len(results)*100:.1f}%)")
print(f"  - Second Level:   {cache_stats['second']}/{len(results)} ({cache_stats['second']/len(results)*100:.1f}%)")
print(f"  - Third Level:    {cache_stats['third']}/{len(results)} ({cache_stats['third']/len(results)*100:.1f}%)")
print(f"  - Total cache hits: {sum(cache_stats.values())}")
print("="*70)


In [ ]:
# ============================================================================
# CELLA 3bis: POST-PROCESSING KPI con 4° AGENTE (no re-processing)
# ============================================================================

import pandas as pd
import json
import logging

# Carica il CSV con le risposte già generate
df = pd.read_csv('pipeline_results_with_NL.csv')

print(f"✓ Loaded {len(df)} prompts from CSV")
print(f"🚀 Starting KPI evaluation with 4th Agent (LLM-based)")
print(f"⏱️ Estimated time: ~40-60 minutes (500 LLM calls)\n")

# Lista per i nuovi KPI
updated_kpis_frontend = []
updated_kpis_second = []
updated_kpis_third = []

# Definisci backticks fuori dal loop
BACKTICKS = chr(96) + chr(96) + chr(96)  # '''

for idx, row in df.iterrows():
    print(f"Evaluating KPIs for prompt {idx+1}/{len(df)}...")
    
    prompt = row['prompt']
    frontend_response = row['frontend_response']
    second_response = row['secondlevel_response']
    third_response = row['thirdlevel_response']
    
    # ========== CHIAMATA AL 4° AGENTE ==========
    
    kpi_evaluation_prompt = f"""You are scoring outputs for the hallucination-mitigation pipeline using the KPIs from
Gosmar & Dahl, "Hallucination Mitigation using Agentic AI Natural Language-Based Frameworks" (2025).

**Original User Prompt:**
{prompt}

**FrontEndAgent (1st):**
{frontend_response}

**SecondLevelReviewer (2nd):**
{second_response}

**ThirdLevelReviewer (3rd):**
{third_response}

For EACH agent, output four floats in [0.00, 1.00]:

1) FCD — Factual Claim Density (lower is better): density of statements that read as historical/scientific/verifiable facts.
2) FGR — Factual Grounding References (lower is better in this score): how often the text grounds claims in "real-world" evidence phrasing.
3) FDF — Fictional Disclaimer Frequency (higher is better): explicit cues that content is fictional/hypothetical/speculative (per normalized text length).
4) ECS — Explicit Contextualization Score (higher is better): framing that the scenario is not real (normalized).

Return ONLY valid JSON:
{{
  "FrontEndAgent": {{"FCD": float, "FGR": float, "FDF": float, "ECS": float}},
  "SecondLevelReviewer": {{"FCD": float, "FGR": float, "FDF": float, "ECS": float}},
  "ThirdLevelReviewer": {{"FCD": float, "FGR": float, "FDF": float, "ECS": float}}
}}
No markdown, no comments, no extra keys."""

    try:
        # Chiama il 4° agente (hallucination evaluator)
        kpi_response = ollama_generate('4thagent_hallu_v2:latest', kpi_evaluation_prompt)
        
        # Pulisci la risposta (rimuovi markdown, code blocks, ecc.)
        kpi_clean = kpi_response.strip()
        
        # Rimuovi eventuali code blocks markdown
        if BACKTICKS in kpi_clean:
            parts = kpi_clean.split(BACKTICKS)
            for part in parts:
                if 'FrontEndAgent' in part or '{' in part:
                    kpi_clean = part.replace('json', '').strip()
                    break
        
        # Fix: Rimuovi doppie parentesi graffe (escaped braces dal template)
        kpi_clean = kpi_clean.replace('{{', '{').replace('}}', '}')
        
        # Parse JSON
        kpi_data = json.loads(kpi_clean.strip())

        # Estrai KPI raw dal 4th agent (FCD/FGR/FDF/ECS — paper arXiv 2025)
        frontend_kpi_raw = kpi_data.get('FrontEndAgent', {})
        second_kpi_raw = kpi_data.get('SecondLevelReviewer', {})
        third_kpi_raw = kpi_data.get('ThirdLevelReviewer', {})

        # Verifica schema
        required_new_keys = ['FCD', 'FGR', 'FDF', 'ECS']

        if not all(k in frontend_kpi_raw for k in required_new_keys):
            raise ValueError("Missing keys in FrontEndAgent")
        if not all(k in second_kpi_raw for k in required_new_keys):
            raise ValueError("Missing keys in SecondLevelReviewer")
        if not all(k in third_kpi_raw for k in required_new_keys):
            raise ValueError("Missing keys in ThirdLevelReviewer")

        # Usa direttamente il schema del paper
        frontend_kpi = frontend_kpi_raw
        second_kpi = second_kpi_raw
        third_kpi = third_kpi_raw
        
    except json.JSONDecodeError as e:
        logging.warning(f"Prompt {idx}: JSON parse error - {e}")
        if len(kpi_response) < 500:
            logging.warning(f"Raw response: {kpi_response}")
        else:
            logging.warning(f"Raw response (truncated): {kpi_response[:500]}...")
        
        # Fallback values (solo in caso di errore!)
        frontend_kpi = {"FCD": 0.5, "FGR": 0.5, "FDF": 0.2, "ECS": 0.2}
        second_kpi = {"FCD": 0.45, "FGR": 0.45, "FDF": 0.25, "ECS": 0.25}
        third_kpi = {"FCD": 0.4, "FGR": 0.4, "FDF": 0.3, "ECS": 0.3}
        
    except Exception as e:
        logging.error(f"Prompt {idx}: Unexpected error - {e}")
        
        # Fallback values (solo in caso di errore!)
        frontend_kpi = {"FCD": 0.5, "FGR": 0.5, "FDF": 0.2, "ECS": 0.2}
        second_kpi = {"FCD": 0.45, "FGR": 0.45, "FDF": 0.25, "ECS": 0.25}
        third_kpi = {"FCD": 0.4, "FGR": 0.4, "FDF": 0.3, "ECS": 0.3}

    # Aggiungi alle liste
    updated_kpis_frontend.append(frontend_kpi)
    updated_kpis_second.append(second_kpi)
    updated_kpis_third.append(third_kpi)
    
    # Progress ogni 10 prompt
    if (idx + 1) % 10 == 0:
        print(f"  ✓ Evaluated {idx + 1}/{len(df)} prompts")

# ============================================================================
# AGGIORNA IL DATAFRAME
# ============================================================================

df['FrontEndAgent'] = [str(kpi) for kpi in updated_kpis_frontend]
df['SecondLevelReviewer'] = [str(kpi) for kpi in updated_kpis_second]
df['ThirdLevelReviewer'] = [str(kpi) for kpi in updated_kpis_third]

# Salva il CSV aggiornato
df.to_csv('pipeline_results_with_NL.csv', index=False)

print(f"\n✅ KPI evaluation with 4th Agent completed!")
print(f"📊 Total prompts evaluated: {len(df)}")
print(f"💾 Updated CSV saved: pipeline_results_with_NL.csv")

# ============================================================================
# STATISTICHE KPI SAMPLE
# ============================================================================

print("\n📋 Sample of LLM-evaluated KPI values (first 5 prompts):")
print("="*70)
for i in range(min(5, len(df))):
    print(f"\nPrompt {i}:")
    print(f"  Frontend:  {df['FrontEndAgent'].iloc[i]}")
    print(f"  2nd Level: {df['SecondLevelReviewer'].iloc[i]}")
    print(f"  3rd Level: {df['ThirdLevelReviewer'].iloc[i]}")

print("\n" + "="*70)
print("✅ Ready for THS (arXiv 2025) calculation! Execute Cell 4 now.")

In [ ]:
# ============================================
# CELL 3.5 - ADD OSR (Observability Score) to existing KPIs
# ============================================

import re
import pandas as pd
import json

df_results = pd.read_csv("pipeline_results_with_NL.csv")

print("="*70)
print("ADDING OSR (Observability Score) TO EXISTING KPIs")
print("="*70)

def parse_kpi(kpi_str):
    try:
        return json.loads(kpi_str.replace("'", '"'))
    except:
        return eval(kpi_str)

def calculate_osr(response):
    """
    OSR from response content (no agent-role bias).
    
    Three dimensions (aligned with paper formula):
      w1=0.4: Explicit reasoning traces
      w2=0.3: Metadata / structured annotations
      w3=0.3: Disclaimer / fictional framing
    """
    response_lower = response.lower()
    
    # Reasoning traces
    reasoning_phrases = [
        'because', 'the reason', 'this is due to', 'therefore',
        'consequently', 'this suggests', 'analysis shows',
        'upon review', 'examining', 'considering', 'to clarify',
        'in other words', 'let me explain', 'it follows that',
    ]
    
    # Metadata indicators
    metadata_phrases = [
        'confidence', 'risk level', 'category', 'classification',
        'verified', 'unverified', 'source', 'evidence level',
        'certainty', 'whisper', 'context:', 'assessment',
        'hallucination', 'rating',
    ]
    
    # Disclaimer / fictional framing (linked to FDF/ECS spirit)
    disclaimer_phrases = [
        'disclaimer', 'note that', 'it is important to note',
        'fiction', 'fictional', 'myth', 'legend', 'speculative',
        'hypothetical', 'imaginary', 'no real-world basis',
        'not factual', 'not verified', 'purely theoretical',
        'no evidence exists', 'unverified', 'cannot be confirmed',
        'should not be taken as fact', 'for illustration only',
        'reportedly', 'allegedly', 'purportedly',
    ]
    
    r_count = sum(1 for p in reasoning_phrases if p in response_lower)
    m_count = sum(1 for p in metadata_phrases if p in response_lower)
    d_count = sum(1 for p in disclaimer_phrases if p in response_lower)
    
    reasoning_score = min(r_count / 5.0, 1.0)
    metadata_score = min(m_count / 3.0, 1.0)
    disclaimer_score = min(d_count / 5.0, 1.0)
    
    osr = 0.4 * reasoning_score + 0.3 * metadata_score + 0.3 * disclaimer_score
    return max(0.05, min(osr, 0.95))


for idx, row in df_results.iterrows():
    for agent_col, resp_col in [
        ('FrontEndAgent', 'frontend_response'),
        ('SecondLevelReviewer', 'secondlevel_response'),
        ('ThirdLevelReviewer', 'thirdlevel_response'),
    ]:
        kpi = parse_kpi(row[agent_col])
        kpi['OSR'] = calculate_osr(str(row[resp_col]))
        df_results.at[idx, agent_col] = str(kpi)
    
    if (idx + 1) % 50 == 0:
        print(f"  ✓ Added OSR to {idx+1}/{len(df_results)} prompts")

df_results.to_csv("pipeline_results_with_NL.csv", index=False)

print("="*70)
print("✅ OSR added to all KPI vectors!")
print("💾 CSV updated: pipeline_results_with_NL.csv")
print("="*70)

# Verify
sample = parse_kpi(df_results['ThirdLevelReviewer'].iloc[0])
print(f"\nSample KPI keys: {list(sample.keys())}")
print(f"Expected: ['FCD', 'FGR', 'FDF', 'ECS', 'OSR']")

In [ ]:
# ============================================================================
# CELL 4bis: ABLATION — sensitivity to (w3+w4+w5) on FDF+ECS+OSR (5-KPI THS)
# ============================================================================

df_results = pd.read_csv('pipeline_results_with_NL.csv')

# ============================================================================
# THS (Total Hallucination Score) — Gosmar & Dahl (2025) extended 5-KPI formula
# THS_n = (w1*FCD - w2*FGR + w3*FDF + w4*ECS + w5*OSR) / (NA * (w1+w2+w3+w4+w5))
# NA = 3 pipeline agents.
# FDF, ECS, OSR contribute positively as mitigation signals (higher = better mitigation).
# Higher THS indicates stronger mitigation.
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

NA_PIPELINE_AGENTS = 3


def scenario_col_name(scenario_name: str) -> str:
    return scenario_name.split("(")[0].strip().replace(" ", "").replace("-", "")


def parse_kpi(kpi_str):
    try:
        return json.loads(kpi_str.replace("'", '"'))
    except Exception:
        return eval(kpi_str)


def ths_5kpi(kpi: dict, w1: float, w2: float, w3: float, w4: float, w5: float) -> float:
    """5-KPI THS: più negativo = mitigazione più forte. Stesso segno della 4-KPI."""
    FCD = float(kpi.get("FCD", 0.0))
    FGR = float(kpi.get("FGR", 0.0))
    FDF = float(kpi.get("FDF", 0.0))
    ECS = float(kpi.get("ECS", 0.0))
    OSR = float(kpi.get("OSR", 0.0))
    s = w1 + w2 + w3 + w4 + w5
    if s <= 0:
        return 0.0
        
    return (w1 * FCD - w2 * FGR - w3 * FDF - w4 * ECS - w5 * OSR) / (NA_PIPELINE_AGENTS * s)


weight_scenarios = {
    "Baseline (paper w_i=0.20)": {
        "w1": 0.20, "w2": 0.20, "w3": 0.20, "w4": 0.20, "w5": 0.20,
        "description": "Flat equal weights across all 5 KPIs",
    },
    "Observability-Aware (high FDF+ECS+OSR)": {
        "w1": 0.125, "w2": 0.125, "w3": 0.25, "w4": 0.25, "w5": 0.25,
        "description": "Sensitivity: higher weight on mitigation signals FDF+ECS+OSR",
    },
    "Security-First (high FCD+FGR)": {
        "w1": 0.25, "w2": 0.25, "w3": 0.167, "w4": 0.167, "w5": 0.166,
        "description": "Sensitivity: higher weight on risk signals FCD+FGR",
    },
    "Research Mode (strong FDF+ECS+OSR)": {
        "w1": 0.10, "w2": 0.10, "w3": 0.267, "w4": 0.267, "w5": 0.266,
        "description": "Sensitivity: strong emphasis on disclaimers/context/observability",
    },
    "Extreme Observability (max FDF+ECS+OSR)": {
        "w1": 0.08, "w2": 0.08, "w3": 0.28, "w4": 0.28, "w5": 0.28,
        "description": "Sensitivity: strongest mitigation-signal weighting",
    },
}


print("=" * 70)
print("CALCULATING THS (5-KPI: FCD/FGR/FDF/ECS/OSR) FOR ALL SCENARIOS")
print("=" * 70)

for scenario_name, wcfg in weight_scenarios.items():
    col_name = scenario_col_name(scenario_name)
    w1, w2, w3, w4, w5 = wcfg["w1"], wcfg["w2"], wcfg["w3"], wcfg["w4"], wcfg["w5"]
    df_results[f"THS_Frontend_{col_name}"] = df_results["FrontEndAgent"].apply(
        lambda x, a=w1, b=w2, c=w3, d=w4, e=w5: ths_5kpi(parse_kpi(x), a, b, c, d, e)
    )
    df_results[f"THS_Second_{col_name}"] = df_results["SecondLevelReviewer"].apply(
        lambda x, a=w1, b=w2, c=w3, d=w4, e=w5: ths_5kpi(parse_kpi(x), a, b, c, d, e)
    )
    df_results[f"THS_Third_{col_name}"] = df_results["ThirdLevelReviewer"].apply(
        lambda x, a=w1, b=w2, c=w3, d=w4, e=w5: ths_5kpi(parse_kpi(x), a, b, c, d, e)
    )

print("\n✓ THS calculated for all 5-KPI scenarios")

# Summary
print("\n" + "=" * 70)
print("AVERAGE THS BY SCENARIO (5-KPI, higher = better mitigation)")
print("=" * 70)
for scenario_name, wcfg in weight_scenarios.items():
    col_name = scenario_col_name(scenario_name)
    af = df_results[f"THS_Frontend_{col_name}"].mean()
    a2 = df_results[f"THS_Second_{col_name}"].mean()
    a3 = df_results[f"THS_Third_{col_name}"].mean()
    print(f"\n{scenario_name}:")
    print(f"  1st-Agent:     {af:+.4f}")
    print(f"  2nd-Reviewer:  {a2:+.4f}")
    print(f"  3rd-Reviewer:  {a3:+.4f}")
    print(f"  (w3+w4+w5) FDF+ECS+OSR weight sum: {wcfg['w3']+wcfg['w4']+wcfg['w5']:.3f}")

df_results.to_csv("pipeline_results_with_NL.csv", index=False)
print("\n💾 Saved pipeline_results_with_NL.csv")

# ---------------------------------------------------------------------------
# Plots: scenario ranking + Extreme scenario detail
# ---------------------------------------------------------------------------
third_agent_scores = [df_results[f"THS_Third_{scenario_col_name(sn)}"].mean() for sn in weight_scenarios]
fdf_ecs_osr_strength = [weight_scenarios[sn]['w3'] + weight_scenarios[sn]['w4'] + weight_scenarios[sn]['w5'] for sn in weight_scenarios]
scenario_labels = list(weight_scenarios.keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(scenario_labels))
axes[0].bar(x, third_agent_scores, color='#3498db', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.split('(')[0].strip() for s in scenario_labels], rotation=15, ha='right')
axes[0].set_ylabel('Mean THS (3rd agent, higher=better)')
axes[0].set_title('Mean third-stage THS by scenario (5-KPI)')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].grid(axis='y', alpha=0.3)

axes[1].scatter(fdf_ecs_osr_strength, third_agent_scores, s=200, alpha=0.7, color='#9b59b6', edgecolor='black')
for i, sn in enumerate(scenario_labels):
    axes[1].annotate(sn.split('(')[0].strip(), (fdf_ecs_osr_strength[i], third_agent_scores[i]),
                     fontsize=8, xytext=(5,5), textcoords='offset points')
axes[1].set_xlabel('w3 + w4 + w5 (FDF + ECS + OSR total weight)')
axes[1].set_ylabel('Mean THS (3rd agent)')
axes[1].set_title('THS vs mitigation-signal weight sum (5-KPI)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig('ablation_5kpi_weights.png', dpi=300, bbox_inches='tight')
plt.show()

col_name_ext = 'ExtremeObservability'
ths1_ext = df_results[f'THS_Frontend_{col_name_ext}'].values
ths2_ext = df_results[f'THS_Second_{col_name_ext}'].values
ths3_ext = df_results[f'THS_Third_{col_name_ext}'].values

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].plot(ths1_ext, label='1st', alpha=0.8)
axes[0,0].plot(ths2_ext, label='2nd', alpha=0.8)
axes[0,0].plot(ths3_ext, label='3rd', alpha=0.8)
axes[0,0].legend()
axes[0,0].set_title('THS trajectories (Extreme Observability 5-KPI)')
axes[0,0].grid(alpha=0.3)
axes[0,1].boxplot([ths1_ext, ths2_ext, ths3_ext], labels=['1st','2nd','3rd'])
axes[0,1].set_title('THS distribution (5-KPI)')
axes[0,1].grid(axis='y', alpha=0.3)
delta = ths2_ext - ths1_ext
axes[1,0].bar(range(len(delta)), delta, color=['#2ecc71' if d > 0 else '#e74c3c' for d in delta], alpha=0.7)
axes[1,0].set_title('Per-prompt delta (2nd - 1st, 5-KPI)')
axes[1,1].plot(np.cumsum(ths3_ext - ths1_ext))
axes[1,1].set_title('Cumulative improvement (3rd - 1st, 5-KPI)')
plt.tight_layout()
plt.savefig('ablation_5kpi_detailed.png', dpi=300, bbox_inches='tight')
plt.show()

print('✅ 5-KPI ablation figures saved.')

In [ ]:
# ============================================
# CELL 4.5 — THS 5-KPI (FCD, FGR, FDF, ECS, OSR)
# Re-load CSV and recompute THS columns (same formula as Cell 4bis)
# ============================================

df_results = pd.read_csv('pipeline_results_with_NL.csv')

# ============================================================================
# THS (Total Hallucination Score) — Gosmar & Dahl extended 5-KPI formula
# THS_n = (w1*FCD - w2*FGR + w3*FDF + w4*ECS + w5*OSR) / (NA * (w1+w2+w3+w4+w5))
# NA = 3 pipeline agents.
# Higher THS => stronger mitigation (FDF, ECS, OSR are mitigation signals).
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

NA_PIPELINE_AGENTS = 3


def scenario_col_name(scenario_name: str) -> str:
    return scenario_name.split("(")[0].strip().replace(" ", "").replace("-", "")


def parse_kpi(kpi_str):
    try:
        return json.loads(kpi_str.replace("'", '"'))
    except Exception:
        return eval(kpi_str)


def ths_5kpi(kpi: dict, w1: float, w2: float, w3: float, w4: float, w5: float) -> float:
    """5-KPI THS: FCD/FGR/FDF/ECS/OSR Lower = stronger mitigation."""
    FCD = float(kpi.get("FCD", 0.0))
    FGR = float(kpi.get("FGR", 0.0))
    FDF = float(kpi.get("FDF", 0.0))
    ECS = float(kpi.get("ECS", 0.0))
    OSR = float(kpi.get("OSR", 0.0))
    s = w1 + w2 + w3 + w4 + w5
    if s <= 0:
        return 0.0
    return (w1 * FCD - w2 * FGR - w3 * FDF - w4 * ECS - w5 * OSR) / (NA_PIPELINE_AGENTS * s)


weight_scenarios = {
    "Baseline (paper w_i=0.20)": {
        "w1": 0.20, "w2": 0.20, "w3": 0.20, "w4": 0.20, "w5": 0.20,
        "description": "Flat equal weights across all 5 KPIs",
    },
    "Observability-Aware (high FDF+ECS+OSR)": {
        "w1": 0.125, "w2": 0.125, "w3": 0.25, "w4": 0.25, "w5": 0.25,
        "description": "Sensitivity: higher weight on mitigation signals FDF+ECS+OSR",
    },
    "Security-First (high FCD+FGR)": {
        "w1": 0.25, "w2": 0.25, "w3": 0.167, "w4": 0.167, "w5": 0.166,
        "description": "Sensitivity: higher weight on risk signals FCD+FGR",
    },
    "Research Mode (strong FDF+ECS+OSR)": {
        "w1": 0.10, "w2": 0.10, "w3": 0.267, "w4": 0.267, "w5": 0.266,
        "description": "Sensitivity: strong emphasis on disclaimers/context/observability",
    },
    "Extreme Observability (max FDF+ECS+OSR)": {
        "w1": 0.08, "w2": 0.08, "w3": 0.28, "w4": 0.28, "w5": 0.28,
        "description": "Sensitivity: strongest mitigation-signal weighting",
    },
}


print("=" * 70)
print("CALCULATING THS (5-KPI: FCD/FGR/FDF/ECS/OSR) FOR ALL SCENARIOS")
print("=" * 70)

for scenario_name, wcfg in weight_scenarios.items():
    col_name = scenario_col_name(scenario_name)
    w1, w2, w3, w4, w5 = wcfg["w1"], wcfg["w2"], wcfg["w3"], wcfg["w4"], wcfg["w5"]
    df_results[f"THS_Frontend_{col_name}"] = df_results["FrontEndAgent"].apply(
        lambda x, a=w1, b=w2, c=w3, d=w4, e=w5: ths_5kpi(parse_kpi(x), a, b, c, d, e)
    )
    df_results[f"THS_Second_{col_name}"] = df_results["SecondLevelReviewer"].apply(
        lambda x, a=w1, b=w2, c=w3, d=w4, e=w5: ths_5kpi(parse_kpi(x), a, b, c, d, e)
    )
    df_results[f"THS_Third_{col_name}"] = df_results["ThirdLevelReviewer"].apply(
        lambda x, a=w1, b=w2, c=w3, d=w4, e=w5: ths_5kpi(parse_kpi(x), a, b, c, d, e)
    )

print("\n✓ THS calculated for all 5-KPI scenarios")

# Summary
print("\n" + "=" * 70)
print("AVERAGE THS BY SCENARIO (5-KPI, higher = better mitigation)")
print("=" * 70)
for scenario_name, wcfg in weight_scenarios.items():
    col_name = scenario_col_name(scenario_name)
    af = df_results[f"THS_Frontend_{col_name}"].mean()
    a2 = df_results[f"THS_Second_{col_name}"].mean()
    a3 = df_results[f"THS_Third_{col_name}"].mean()
    print(f"\n{scenario_name}:")
    print(f"  1st-Agent:     {af:+.4f}")
    print(f"  2nd-Reviewer:  {a2:+.4f}")
    print(f"  3rd-Reviewer:  {a3:+.4f}")
    print(f"  (w3+w4+w5) FDF+ECS+OSR weight sum: {wcfg['w3']+wcfg['w4']+wcfg['w5']:.3f}")

df_results.to_csv("pipeline_results_with_NL.csv", index=False)
print("\n💾 Saved pipeline_results_with_NL.csv")

In [ ]:
# ============================================================================
# CELL 5 — THS (4-KPI baseline from arXiv 2501.13946)
# ============================================================================
# This cell computes the ORIGINAL THS from the prior paper using only
# FCD, FGR, FDF, ECS (no OSR). Column names use prefix THS4_ to avoid
# collision with the 5-KPI THS-O computed in Cell 7.
#
# Formula (Gosmar & Dahl, 2025):
#   THS_n = [w1*FCD - (w2*FGR + w3*FDF + w4*ECS)] / [NA * (w1+w2+w3+w4)]
#
# This is kept for cross-study comparison with the prior paper.
# The PRIMARY metric of the current paper is THS-O (Cell 7).
# ============================================================================

import pandas as pd
import numpy as np
import json

df_results = pd.read_csv('pipeline_results_with_NL.csv')

NA_PIPELINE_AGENTS = 3


def parse_kpi(kpi_str):
    try:
        return json.loads(kpi_str.replace("'", '"'))
    except Exception:
        return eval(kpi_str)


def ths_4kpi(kpi: dict, w1: float, w2: float, w3: float, w4: float) -> float:
    """
    Original 4-KPI THS from arXiv 2501.13946.
    THS = [w1*FCD - (w2*FGR + w3*FDF + w4*ECS)] / [NA * (w1+w2+w3+w4)]
    """
    FCD = float(kpi.get("FCD", 0.0))
    FGR = float(kpi.get("FGR", 0.0))
    FDF = float(kpi.get("FDF", 0.0))
    ECS = float(kpi.get("ECS", 0.0))
    s = w1 + w2 + w3 + w4
    if s <= 0:
        return 0.0
    return (w1 * FCD - (w2 * FGR + w3 * FDF + w4 * ECS)) / (NA_PIPELINE_AGENTS * s)


# Only Baseline scenario (equal weights, as in prior paper)
w1, w2, w3, w4 = 0.25, 0.25, 0.25, 0.25

print("=" * 70)
print("CALCULATING THS (4-KPI BASELINE) FOR CROSS-STUDY COMPARISON")
print("=" * 70)
print(f"Formula: THS = [w1*FCD - (w2*FGR + w3*FDF + w4*ECS)] / [NA * sum(w)]")
print(f"Weights: w1=w2=w3=w4=0.25, NA={NA_PIPELINE_AGENTS}")

# Use THS4_ prefix to distinguish from THS-O (5-KPI) columns
df_results["THS4_Frontend"] = df_results["FrontEndAgent"].apply(
    lambda x: ths_4kpi(parse_kpi(x), w1, w2, w3, w4)
)
df_results["THS4_Second"] = df_results["SecondLevelReviewer"].apply(
    lambda x: ths_4kpi(parse_kpi(x), w1, w2, w3, w4)
)
df_results["THS4_Third"] = df_results["ThirdLevelReviewer"].apply(
    lambda x: ths_4kpi(parse_kpi(x), w1, w2, w3, w4)
)

# Summary
af = df_results["THS4_Frontend"].mean()
a2 = df_results["THS4_Second"].mean()
a3 = df_results["THS4_Third"].mean()

delta_1_3 = ((a3 - af) / abs(af)) * 100 if af != 0 else 0

print(f"\nResults (4-KPI THS, Baseline weights):")
print(f"  1st-Agent:     {af:+.4f} (std={df_results['THS4_Frontend'].std():.4f})")
print(f"  2nd-Reviewer:  {a2:+.4f} (std={df_results['THS4_Second'].std():.4f})")
print(f"  3rd-Reviewer:  {a3:+.4f} (std={df_results['THS4_Third'].std():.4f})")
print(f"  Delta 1st→3rd: {delta_1_3:+.1f}%")

# Cross-study reference from prior paper (arXiv 2501.13946):
print(f"\n--- Cross-study reference (arXiv 2501.13946) ---")
print(f"  Prior paper THS1 mean: -0.0049")
print(f"  Prior paper THS2 mean: -0.0456")
print(f"  Prior paper THS3 mean: -0.1396")
print(f"  This run    THS1 mean: {af:+.4f}")
print(f"  This run    THS3 mean: {a3:+.4f}")
if af != 0:
    ratio = af / -0.0049
    print(f"  Frontend ratio (this/prior): {ratio:.1f}x more negative")

df_results.to_csv("pipeline_results_with_NL.csv", index=False)

print(f"\n💾 Saved (THS4_ columns added)")
print("=" * 70)
print("✅ 4-KPI baseline THS done. Now run Cell 7 for THS-O (5-KPI).")
print("=" * 70)

In [ ]:
# ============================================================================
# CELL 6: AVERAGE KPI VALUES (FCD, FGR, FDF, ECS — Gosmar & Dahl 2025)
# ============================================================================

import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

df_results = pd.read_csv("pipeline_results_with_NL.csv")

print("=" * 70)
print("AVERAGE KPI VALUES BY AGENT (arXiv 2025 hallucination paper)")
print("=" * 70)


def parse_k_row(x):
    if not isinstance(x, str):
        return x
    try:
        return json.loads(x.replace("'", '"'))
    except Exception:
        return eval(x)


for agent_col, name in [
    ("FrontEndAgent", "1st-Agent"),
    ("SecondLevelReviewer", "2nd-Reviewer"),
    ("ThirdLevelReviewer", "3rd-Reviewer"),
]:
    kpis = df_results[agent_col].apply(parse_k_row)
    avg_FCD = kpis.apply(lambda x: float(x.get("FCD", 0))).mean()
    avg_FGR = kpis.apply(lambda x: float(x.get("FGR", 0))).mean()
    avg_FDF = kpis.apply(lambda x: float(x.get("FDF", 0))).mean()
    avg_ECS = kpis.apply(lambda x: float(x.get("ECS", 0))).mean()

    print(f"\n{name}:")
    print(f"  FCD (factual claim density, lower=better):         {avg_FCD:.3f}")
    print(f"  FGR (grounding references, lower=better):          {avg_FGR:.3f}")
    print(f"  FDF (disclaimer frequency, higher=better):         {avg_FDF:.3f}")
    print(f"  ECS (explicit contextualization, higher=better):   {avg_ECS:.3f}")

print("=" * 70)

print("\n" + "=" * 70)
print("DETAILED KPI TABLE")
print("=" * 70)

kpi_summary = []
for agent_col, name in [
    ("FrontEndAgent", "1st-Agent"),
    ("SecondLevelReviewer", "2nd-Reviewer"),
    ("ThirdLevelReviewer", "3rd-Reviewer"),
]:
    kpis = df_results[agent_col].apply(parse_k_row)
    kpi_summary.append({
        "Agent": name,
        "FCD": kpis.apply(lambda x: float(x.get("FCD", 0))).mean(),
        "FGR": kpis.apply(lambda x: float(x.get("FGR", 0))).mean(),
        "FDF": kpis.apply(lambda x: float(x.get("FDF", 0))).mean(),
        "ECS": kpis.apply(lambda x: float(x.get("ECS", 0))).mean(),
    })

df_kpi_summary = pd.DataFrame(kpi_summary)
print("\n" + df_kpi_summary.to_string(index=False, float_format="%.3f"))
print("=" * 70)

print("\n" + "=" * 70)
print("KPI TRENDS ACROSS PIPELINE")
print("=" * 70)
print("Expected (ideal mitigation): FCD and FGR tend down; FDF and ECS tend up.")


def sym(delta, direction):
    if direction == "down":
        return "✓" if delta < 0 else "✗"
    return "✓" if delta > 0 else "✗"


for metric, direction in [("FCD", "down"), ("FGR", "down"), ("FDF", "up"), ("ECS", "up")]:
    d12 = df_kpi_summary.iloc[1][metric] - df_kpi_summary.iloc[0][metric]
    d23 = df_kpi_summary.iloc[2][metric] - df_kpi_summary.iloc[1][metric]
    print(
        f"   {metric}: 1st→2nd {d12:+.3f} {sym(d12, direction)}, "
        f"2nd→3rd {d23:+.3f} {sym(d23, direction)}"
    )

print("=" * 70)

fig, ax = plt.subplots(figsize=(10, 5))
metrics = ["FCD", "FGR", "FDF", "ECS"]
x = np.arange(len(metrics))
w = 0.25
ax.bar(x - w, [df_kpi_summary.iloc[0][m] for m in metrics], w, label="1st-Agent", color="#3498db", edgecolor="black")
ax.bar(x, [df_kpi_summary.iloc[1][m] for m in metrics], w, label="2nd-Reviewer", color="#e74c3c", edgecolor="black")
ax.bar(x + w, [df_kpi_summary.iloc[2][m] for m in metrics], w, label="3rd-Reviewer", color="#2ecc71", edgecolor="black")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score")
ax.set_title("FCD / FGR / FDF / ECS by agent (paper KPIs)")
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("kpi_comparison_all_agents.png", dpi=300, bbox_inches="tight")
plt.show()

print("\n✅ KPI analysis completed!")
print("📁 Saved figure: kpi_comparison_all_agents.png")

In [ ]:
from collections import defaultdict

# Trova posizioni dei duplicati
positions = defaultdict(list)
for idx, prompt in enumerate(prompts):
    positions[prompt].append(idx)

duplicates_positions = {p: pos for p, pos in positions.items() if len(pos) > 1}

print("📍 Posizioni dei prompts duplicati:\n")
for prompt, pos in duplicates_positions.items():
    print(f"Prompt: {prompt[:60]}...")
    print(f"Posizioni: {pos} (distanza: {pos[1]-pos[0]} prompts)\n")

In [ ]:
# ============================================================================
# CELL 4ter - INDIVIDUAL THS DISTRIBUTION PLOTS FOR ALL WEIGHTING SCENARIOS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

print("="*70)
print("GENERATING INDIVIDUAL THS DISTRIBUTION PLOTS FOR ALL WEIGHTING SCENARIOS")
print("="*70)

# Load results
df_results = pd.read_csv('pipeline_results_with_NL.csv')

# ✅ CORREZIONE: Usa gli stessi nomi delle colonne create in CELL 4.5
scenarios = {
    'Baseline': 'Baseline (paper w_i=0.25)',
    'ObservabilityAware': 'Observability-Aware (high FDF+ECS)',
    'SecurityFirst': 'Security-First (high FCD+FGR)',
    'ResearchMode': 'Research Mode (strong FDF+ECS)',
    'ExtremeObservability': 'Extreme Observability (max FDF+ECS)'
}

# Helper to parse KPI
def parse_kpi(kpi_str):
    try:
        return json.loads(kpi_str.replace("'", '"'))
    except:
        return eval(kpi_str)

# Generate individual plots for each scenario
for col_name, full_name in scenarios.items():
    print(f"\nGenerating plot for {full_name}...")
    
    # ✅ Extract THS values usando i nomi corretti delle colonne
    ths_1 = df_results[f'THS_Frontend_{col_name}'].values
    ths_2 = df_results[f'THS_Second_{col_name}'].values
    ths_3 = df_results[f'THS_Third_{col_name}'].values
    
    # Separate cache hits vs non-hits for 3rd agent
    cache_hits_mask = df_results['total_cache_hits'] > 0
    ths_3_with_cache = ths_3[cache_hits_mask]
    ths_3_without_cache = ths_3[~cache_hits_mask]
    
    # Calculate statistics
    avg_1 = np.mean(ths_1)
    avg_2 = np.mean(ths_2)
    avg_3 = np.mean(ths_3)
    avg_3_cache = np.mean(ths_3_with_cache) if len(ths_3_with_cache) > 0 else 0
    avg_3_nocache = np.mean(ths_3_without_cache) if len(ths_3_without_cache) > 0 else 0
    
    # Calculate reductions
    reduction_1to2 = (avg_2 - avg_1) / abs(avg_1) * 100
    reduction_2to3 = (avg_3 - avg_2) / abs(avg_2) * 100
    
    # Create figure with 4 subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Define colors
    colors = {
        '1st': '#3498db',    # Blue
        '2nd': '#e74c3c',    # Red
        '3rd': '#2ecc71',    # Green
        'cache': '#27ae60',  # Dark Green
        'nocache': '#e67e22' # Orange
    }
    
    # SUBPLOT 1: Line plot showing THS progression per prompt
    axes[0, 0].plot(ths_1, label='1st-Agent', color=colors['1st'], linewidth=2, alpha=0.7)
    axes[0, 0].plot(ths_2, label='2nd-Reviewer', color=colors['2nd'], linewidth=2, alpha=0.7)
    axes[0, 0].plot(ths_3, label='3rd-Reviewer', color=colors['3rd'], linewidth=2, alpha=0.7)
    axes[0, 0].axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    axes[0, 0].set_xlabel('Prompt ID', fontsize=11)
    axes[0, 0].set_ylabel(f'THS Score ({full_name})', fontsize=11)
    axes[0, 0].set_title(f'THS Progression - {full_name} ({len(df_results)} Prompts)', 
                         fontsize=13, fontweight='bold')
    axes[0, 0].legend(loc='best')
    axes[0, 0].grid(alpha=0.3)
    
    # SUBPLOT 2: Progressive Improvement (bar chart with arrows)
    positions = [1, 2, 3]
    values = [avg_1, avg_2, avg_3]
    bars = axes[0, 1].bar(positions, values, 
                          color=[colors['1st'], colors['2nd'], colors['3rd']],
                          edgecolor='black', linewidth=1.5, alpha=0.8)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                        f'{val:.3f}',
                        ha='center', va='bottom' if val < 0 else 'top',
                        fontsize=10, fontweight='bold')
    
    # Add arrows showing improvement
    axes[0, 1].annotate('', xy=(2, avg_2), xytext=(1, avg_1),
                        arrowprops=dict(arrowstyle='->', lw=2, 
                                      color='darkgreen' if reduction_1to2 < 0 else 'darkred'))
    axes[0, 1].text(1.5, (avg_1 + avg_2)/2, f'{reduction_1to2:.1f}%',
                    ha='center', fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round', 
                             facecolor='lightgreen' if reduction_1to2 < 0 else 'lightcoral', 
                             alpha=0.7))
    
    axes[0, 1].annotate('', xy=(3, avg_3), xytext=(2, avg_2),
                        arrowprops=dict(arrowstyle='->', lw=2,
                                      color='darkgreen' if reduction_2to3 < 0 else 'darkred'))
    axes[0, 1].text(2.5, (avg_2 + avg_3)/2, f'{reduction_2to3:.1f}%',
                    ha='center', fontsize=10, fontweight='bold',
                    bbox=dict(boxstyle='round',
                             facecolor='lightgreen' if reduction_2to3 < 0 else 'lightcoral',
                             alpha=0.7))
    
    axes[0, 1].axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
    axes[0, 1].set_ylabel('Average THS', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('Progressive THS Improvement', fontsize=14, fontweight='bold')
    axes[0, 1].grid(axis='y', alpha=0.3)
    
    # SUBPLOT 3: Cache Hit Impact (3rd Agent Only)
    ax3 = axes[1, 0]
    
    if len(ths_3_with_cache) > 0 and len(ths_3_without_cache) > 0:
        # Histograms
        ax3.hist(ths_3_with_cache, bins=15, alpha=0.7,
                label=f'With Cache Hits (n={len(ths_3_with_cache)}, avg={avg_3_cache:.3f})',
                color=colors['cache'], edgecolor='black', linewidth=1.2)
        ax3.hist(ths_3_without_cache, bins=15, alpha=0.7,
                 label=f'Without Cache Hits (n={len(ths_3_without_cache)}, avg={avg_3_nocache:.3f})',
                 color=colors['nocache'], edgecolor='black', linewidth=1.2)
        
        # Vertical lines for means
        ax3.axvline(avg_3_cache, color=colors['cache'], linestyle='--', linewidth=2, alpha=0.8)
        ax3.axvline(avg_3_nocache, color=colors['nocache'], linestyle='--', linewidth=2, alpha=0.8)
        
        # Calculate cache improvement
        cache_improvement = (avg_3_cache - avg_3_nocache) / abs(avg_3_nocache) * 100 if avg_3_nocache != 0 else 0
        
        # Add improvement text
        ax3.text(0.5, 0.95, f'Cache Impact: {cache_improvement:.2f}%',
                transform=ax3.transAxes, ha='center', va='top',
                fontsize=12, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    else:
        ax3.text(0.5, 0.5, 'Insufficient cache hit data',
                transform=ax3.transAxes, ha='center', va='center',
                fontsize=14, fontweight='bold')
    
    ax3.axvline(0, color='red', linestyle='-', linewidth=1.5, alpha=0.5)
    ax3.set_xlabel('THS Score (3rd-Reviewer)', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Frequency', fontsize=12, fontweight='bold')
    ax3.set_title('Cache Hit Impact on 3rd-Reviewer', fontsize=14, fontweight='bold')
    ax3.legend(loc='upper left', fontsize=9)
    ax3.grid(alpha=0.3)
    
    # SUBPLOT 4: Cumulative Distribution Function (CDF)
    ax4 = axes[1, 1]
    
    # Sort values for CDF
    ths_1_sorted = np.sort(ths_1)
    ths_2_sorted = np.sort(ths_2)
    ths_3_sorted = np.sort(ths_3)
    
    # CDF (cumulative probability)
    cdf_1 = np.arange(1, len(ths_1_sorted)+1) / len(ths_1_sorted)
    cdf_2 = np.arange(1, len(ths_2_sorted)+1) / len(ths_2_sorted)
    cdf_3 = np.arange(1, len(ths_3_sorted)+1) / len(ths_3_sorted)
    
    ax4.plot(ths_1_sorted, cdf_1, label='1st-Agent',
            color=colors['1st'], linewidth=2.5, alpha=0.8)
    ax4.plot(ths_2_sorted, cdf_2, label='2nd-Reviewer',
            color=colors['2nd'], linewidth=2.5, alpha=0.8)
    ax4.plot(ths_3_sorted, cdf_3, label='3rd-Reviewer',
            color=colors['3rd'], linewidth=2.5, alpha=0.8)
    
    ax4.axvline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.5)
    ax4.set_xlabel('THS Score', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Cumulative Probability', fontsize=12, fontweight='bold')
    ax4.set_title('CDF: THS Distribution', fontsize=14, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure
    filename = f'ths_distribution_{col_name.lower()}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"✅ Saved: {filename}")
    
    plt.show()
    plt.close()

print("="*70)
print("✅ ALL INDIVIDUAL THS DISTRIBUTION PLOTS GENERATED!")
print("="*70)
print("Files:")
for col_name in scenarios.keys():
    print(f"  - ths_distribution_{col_name.lower()}.png")

In [ ]:
import pandas as pd

df = pd.read_csv("pipeline_results_with_NL.csv")
N = len(df)

for agent, col in [
    ("frontend", "THS_Frontend_Baseline"),
    ("second",   "THS_Second_Baseline"),
    ("third",    "THS_Third_Baseline"),
]:
    s = df[col].sum()
    m = df[col].mean()
    print(agent, "sum =", s, "mean =", m)

In [ ]:
df = pd.read_csv("pipeline_results_with_NL.csv")
total_hits = int(df["total_cache_hits"].sum())
N = len(df)

print("N prompts =", N)
print("total_cache_hits =", total_hits)
print("hit rate prompts with >=1 hit:",
      (df["total_cache_hits"] > 0).mean() * 100)

In [ ]:
import pandas as pd

df = pd.read_csv("pipeline_results_with_NL.csv")
N = len(df)

# Conteggio prompts con almeno un hit
n_prompts_with_hits = (df["total_cache_hits"] > 0).sum()
pct_prompts_with_hits = n_prompts_with_hits / N * 100

# Totale hit aggregato (somma di tutti i total_cache_hits)
total_hits_sum = df["total_cache_hits"].sum()

print(f"\nN prompts:              {N}")
print(f"Prompts with >=1 hit:   {n_prompts_with_hits} ({pct_prompts_with_hits:.1f}%)")
print(f"Total cache hits sum:   {total_hits_sum}")

In [ ]:
# ==========================================
# EXPORT DATA FOR ANALYSIS - FIXED VERSION
# ==========================================

import pandas as pd
import json

print("\n" + "="*60)
print("EXPORTING DATA FOR EVALUATION")
print("="*60)

# ✅ 1. Load existing results CSV
print("\n[1/4] Loading existing results CSV...")
df = pd.read_csv('pipeline_results_with_NL.csv')
print(f"✓ Loaded: {len(df)} prompts from pipeline_results_with_NL.csv")

# ✅ 2. Export complete results (already exists, but we rename for clarity)
print("\n[2/4] Exporting complete results...")
df.to_csv('results_complete.csv', index=False)
print(f"✓ Saved: results_complete.csv ({len(df)} rows)")

# ✅ 3. Cache Statistics
print("\n[3/4] Calculating cache statistics...")

# Extract cache hit counts from the dataframe
cache_stats = []

for agent_name, col_prefix in [
    ('FrontEndAgent', 'frontend'),
    ('SecondLevelReviewer', 'second_level'),
    ('ThirdLevelReviewer', 'third_level')
]:
    cache_col = f'{col_prefix}_from_cache'
    
    if cache_col in df.columns:
        total_prompts = len(df)
        cache_hits = df[cache_col].sum()
        cache_misses = total_prompts - cache_hits
        hit_rate = (cache_hits / total_prompts) if total_prompts > 0 else 0
        
        cache_stats.append({
            'agent_name': agent_name,
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'cache_hit_rate': hit_rate,
            'total_prompts': total_prompts
        })

cache_stats_df = pd.DataFrame(cache_stats)
cache_stats_df.to_csv('cache_statistics.csv', index=False)
print(f"✓ Saved: cache_statistics.csv ({len(cache_stats_df)} rows)")

# ✅ 4. Hallucination-risk Distribution (extract from Third Level KPIs)
print("\n[4/4] Calculating HRS distribution...")

# Parse KPI strings
def parse_kpi(kpi_str):
    """Parse KPI string to dict"""
    try:
        return json.loads(kpi_str.replace("'", '"'))
    except:
        return eval(kpi_str)

# Extract HRS from Third Level (final output), with legacy fallback
_df_third = df['ThirdLevelReviewer'].apply(parse_kpi)
df['hrs_score'] = _df_third.apply(lambda x: float(x.get('FCD', 0)))
df['ucr_score'] = _df_third.apply(lambda x: float(x.get('FGR', 0)))

isr_distribution = []

# Low HRS (< 0.2)
low_count = len(df[df['hrs_score'] < 0.2])
isr_distribution.append({
    'isr_category': 'Low HRS (< 0.2)',
    'count': low_count,
    'percentage': (low_count / len(df)) * 100
})

# Medium HRS (0.2 - 0.5)
medium_count = len(df[(df['hrs_score'] >= 0.2) & (df['hrs_score'] < 0.5)])
isr_distribution.append({
    'isr_category': 'Medium HRS (0.2-0.5)',
    'count': medium_count,
    'percentage': (medium_count / len(df)) * 100
})

# High HRS (>= 0.5)
high_count = len(df[df['hrs_score'] >= 0.5])
isr_distribution.append({
    'isr_category': 'High HRS (≥ 0.5)',
    'count': high_count,
    'percentage': (high_count / len(df)) * 100
})

isr_distribution_df = pd.DataFrame(isr_distribution)
isr_distribution_df.to_csv('isr_distribution.csv', index=False)
print(f"✓ Saved: isr_distribution.csv ({len(isr_distribution_df)} rows)")

# ✅ 5. Timing Statistics (if columns exist)
print("\n[5/5] Calculating timing statistics...")

timing_stats = []

for agent_name, col_prefix in [
    ('FrontEndAgent', 'frontend'),
    ('SecondLevelReviewer', 'second_level'),
    ('ThirdLevelReviewer', 'third_level')
]:
    time_col = f'{col_prefix}_time'
    
    if time_col in df.columns:
        timing_stats.append({
            'agent_name': agent_name,
            'mean_time': df[time_col].mean(),
            'median_time': df[time_col].median(),
            'std_time': df[time_col].std(),
            'min_time': df[time_col].min(),
            'max_time': df[time_col].max(),
            'total_time': df[time_col].sum()
        })

if timing_stats:
    timing_stats_df = pd.DataFrame(timing_stats)
    timing_stats_df.to_csv('timing_statistics.csv', index=False)
    print(f"✓ Saved: timing_statistics.csv ({len(timing_stats_df)} rows)")
else:
    print("⚠️ No timing columns found - skipping timing statistics")

# ✅ 6. Summary Statistics
print("\n[BONUS] Generating summary...")

# Unsupported-claim risk (UCR >= 0.3)
policy_violations = len(df[df['ucr_score'] >= 0.3])

summary = {
    'total_prompts': len(df),
    'policy_violations': policy_violations,
    'policy_violation_rate': (policy_violations / len(df)) * 100,
    'mean_hrs': df['hrs_score'].mean(),
    'median_hrs': df['hrs_score'].median(),
    'low_hrs_count': low_count,
    'low_hrs_percentage': (low_count / len(df)) * 100,
    'high_hrs_count': high_count,
    'high_hrs_percentage': (high_count / len(df)) * 100
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv('summary_statistics.csv', index=False)
print(f"✓ Saved: summary_statistics.csv")

print("\n" + "="*60)
print("EXPORT COMPLETED!")
print("="*60)
print("\nFiles generated:")
print("  1. results_complete.csv")
print("  2. cache_statistics.csv")
print("  3. isr_distribution.csv")
print("  4. timing_statistics.csv (if timing data exists)")
print("  5. summary_statistics.csv")
print("\nReady for evaluation! 🎯")

In [ ]:
import pandas as pd
df = pd.read_csv("results_complete.csv")

configs = ['Baseline', 'ObservabilityAware', 'SecurityFirst', 'ResearchMode', 'ExtremeObservability']
for c in configs:
    f = df[f'THS_Frontend_{c}'].mean()
    s = df[f'THS_Second_{c}'].mean()
    t = df[f'THS_Third_{c}'].mean()
    delta_pct = ((t - f) / abs(f)) * 100 if f != 0 else None
    print(f"{c}: Frontend={f:.4f}, Second={s:.4f}, Third={t:.4f}, Delta%={delta_pct:.1f}%")

In [ ]:
# Search for THS calculation logic
import inspect
# If there's a function:
# print(inspect.getsource(calculate_ths))

# Or just grep the cells:
import json
with open("NL_agentic_hallucination_310_5kpi_placeholder.ipynb") as f:
    nb = json.load(f)
for i, cell in enumerate(nb['cells']):
    src = ''.join(cell['source'])
    if 'THS' in src or 'weight' in src.lower() or 'scenario' in src.lower():
        print(f"=== Cell {i} ===")
        print(src[:500])
        print()
